## 1. 90일 기상 센서 로그를 직접 생성해 결측값/이상치를 보정하고 월별 리포트 만들기

CSV 파일 없이 관측 데이터를 직접 만들어 분석하는 과제입니다. 아래 순서대로 진행하세요.

① 랜덤 시계열 Series 생성

np.random.seed(2024)로 시드를 고정하세요.

pd.date_range로 2024년 3월 1일부터 90일간의 날짜 인덱스를 만드세요.

평균 18, 표준편차 4인 정규분포 난수(np.random.normal) 90개를 소수 첫째 자리까지 반올림해 값으로 사용하고, 위 날짜를 인덱스로 갖는 Series s를 만드세요.

head(), describe(), index, dtype을 출력해 구조를 확인하세요.

센서 고장 상황을 인위적으로 만듭니다. np.random.choice로 인덱스 10개를 중복 없이 뽑아 그 위치의 값을 결측값(np.nan)으로 바꾸세요.

이상치도 주입합니다. 결측이 아닌 인덱스 중 5개를 중복 없이 뽑아 그 값을 3배로 만드세요.

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(2024)
date_index = pd.date_range(start='2024-03-01', periods=90)

In [3]:
# 평균 18, 표준편차 4인 정규분포 난수(np.random.normal) 90개 => np.random.normal(loc=18, scale=4, size=90)
values = np.round(np.random.normal(loc=18, scale=4, size=90), 1)

In [5]:
s = pd.Series(values, index=date_index)
s

2024-03-01    24.7
2024-03-02    20.9
2024-03-03    17.2
2024-03-04    17.4
2024-03-05    21.7
              ... 
2024-05-25    20.1
2024-05-26    22.3
2024-05-27    18.4
2024-05-28    15.5
2024-05-29    20.4
Freq: D, Length: 90, dtype: float64

In [6]:
print(s.head())
print(s.describe())
print(s.index)
print(s.dtype)

2024-03-01    24.7
2024-03-02    20.9
2024-03-03    17.2
2024-03-04    17.4
2024-03-05    21.7
Freq: D, dtype: float64
count    90.000000
mean     18.202222
std       3.923996
min       7.500000
25%      15.800000
50%      18.300000
75%      21.350000
max      25.700000
dtype: float64
DatetimeIndex(['2024-03-01', '2024-03-02', '2024-03-03', '2024-03-04',
               '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08',
               '2024-03-09', '2024-03-10', '2024-03-11', '2024-03-12',
               '2024-03-13', '2024-03-14', '2024-03-15', '2024-03-16',
               '2024-03-17', '2024-03-18', '2024-03-19', '2024-03-20',
               '2024-03-21', '2024-03-22', '2024-03-23', '2024-03-24',
               '2024-03-25', '2024-03-26', '2024-03-27', '2024-03-28',
               '2024-03-29', '2024-03-30', '2024-03-31', '2024-04-01',
               '2024-04-02', '2024-04-03', '2024-04-04', '2024-04-05',
               '2024-04-06', '2024-04-07', '2024-04-08', '2024-04-09',
    

In [8]:
random_indices = np.random.choice(len(date_index), size=10, replace=False)  # 중복없이 10개 인덱스 추출

In [ ]:
s.iloc[random_indices] = np.nan
print(s.head(20))

2024-03-01    24.7
2024-03-02    20.9
2024-03-03    17.2
2024-03-04    17.4
2024-03-05    21.7
2024-03-06    22.6
2024-03-07     7.5
2024-03-08    12.7
2024-03-09    19.8
2024-03-10    18.4
2024-03-11    22.2
2024-03-12    24.5
2024-03-13    12.0
2024-03-14    16.9
2024-03-15    22.8
2024-03-16    21.4
2024-03-17    16.3
2024-03-18    17.0
2024-03-19     NaN
2024-03-20    14.9
Freq: D, dtype: float64


In [ ]:
# 1. 결측값(NaN)이 아닌 정상 데이터의 '위치 번호'만 추출
valid_indices = np.where(s.notna())[0]  # s.notna()가 True인 곳의 인덱스 추출

# 2. 정상 위치 번호 중 중복 없이 5개 무작위 선택
another_indices = np.random.choice(valid_indices, size=5, replace=False)  # 중복없이 5개 추출

# 3. 선택된 위치(.iloc)의 값을 기존 값의 3배로 변경
s.iloc[another_indices] = s.iloc[another_indices] * 3

print(s.head(20))

2024-03-01    24.7
2024-03-02    20.9
2024-03-03    17.2
2024-03-04    17.4
2024-03-05    21.7
2024-03-06    22.6
2024-03-07     7.5
2024-03-08    12.7
2024-03-09    19.8
2024-03-10    18.4
2024-03-11    22.2
2024-03-12    24.5
2024-03-13    12.0
2024-03-14    16.9
2024-03-15    68.4
2024-03-16    21.4
2024-03-17    16.3
2024-03-18    17.0
2024-03-19     NaN
2024-03-20    14.9
Freq: D, dtype: float64


② 결측값 보정 방식 4가지 비교

결측이 발생한 날짜 목록과 개수를 출력하세요.

다음 4가지 Series를 각각 만드세요. (1) 결측 제거 (2) 0으로 채움 (3) 전체 평균으로 채움 (4) 바로 앞 날짜 값으로 채움

네 결과의 평균과 표준편차를 하나의 DataFrame으로 정리해 비교하고, 시계열 데이터에서 어떤 방식이 가장 부적절한지 주석으로 근거와 함께 적으세요.

이후 단계에서는 (4) 앞값 채움 결과를 사용하세요.

In [ ]:
missing_dates = s.index[random_indices].sort_values()
print(missing_dates)

print(s.isnull().sum())

DatetimeIndex(['2024-03-19', '2024-03-25', '2024-04-08', '2024-04-12',
               '2024-04-16', '2024-04-28', '2024-05-07', '2024-05-15',
               '2024-05-19', '2024-05-22'],
              dtype='datetime64[us]', freq=None)
10


In [ ]:
# 결측 제거
s1 = s.dropna(ignore_index=True)

In [ ]:
# 결측 부분 0으로 채움
s2 = s.fillna(0)

In [ ]:
# 결측 부분 전체평균으로 채움
mean_Series = s.mean()
s3 = s.fillna(mean_Series)

In [ ]:
# 결측 부분 바로 앞 날짜 값으로 채우기
s4 = s.ffill()

In [33]:
comparison_data = {
    '평균 (Mean)': [s1.mean(), s2.mean(), s3.mean(), s4.mean()],
    '표준편차 (Std)': [s1.std(), s2.std(), s3.std(), s4.std()]
}

In [34]:
df = pd.DataFrame(
    comparison_data, 
    index=['(1) 결측 제거', '(2) 0으로 채움', '(3) 전체 평균으로 채움', '(4) 바로 앞 날짜 값으로 채움']
)
df

# 아래의 결과를 바탕으로 해석하면, 시계열 데이터에서 '(2)0으로 채움' 방식이 가장 부적절
# 무의미한 숫자 0이 주입되면 전체 평균이 왜곡되고(평균이 17.7로 가장 낮음), 
# 표준편차가 비정상적으로 커짐(표준편차가 10.6으로 가장 큼)

,평균 (Mean),표준편차 (Std)
(1) 결측 제거,19.976250,9.021104
(2) 0으로 채움,17.756667,10.587337
(3) 전체 평균으로 채움,19.976250,8.499203
(4) 바로 앞 날짜 값으로 채움,19.797778,9.012687


③ 이상치 탐지와 보정

평균 ± (2 × 표준편차)를 상·하한으로 계산해 출력하세요.

조건 색인으로 상한을 넘거나 하한 밑인 날짜와 값을 출력하고, ①에서 주입한 5일과 일치하는지 확인하세요.

np.where를 중첩해 상한 초과는 상한값으로, 하한 미만은 하한값으로 바꾼 Series clean을 만드세요.

보정 전후의 describe()를 비교해 표준편차가 어떻게 변했는지 주석으로 적으세요.

In [ ]:
s = s.ffill()

In [ ]:
upper_limit = s.mean() + (2 * s.std())
lower_limit = s.mean() - (2 * s.std())
print(upper_limit)
print(lower_limit)

37.82315259574868
1.7724029598068753


In [ ]:
print(s[(s > upper_limit) | (s < lower_limit)])    # 이상치 1개가 복사되어, 총 6개 도출  # False

2024-03-15    68.4
2024-03-26    43.5
2024-04-19    41.1
2024-05-12    52.2
2024-05-14    43.2
2024-05-15    43.2
dtype: float64


In [ ]:
cleaned_values = np.where(s > upper_limit, upper_limit, np.where(s < lower_limit, lower_limit, s))
clean = pd.Series(cleaned_values, index=s.index)
clean

2024-03-01    24.7
2024-03-02    20.9
2024-03-03    17.2
2024-03-04    17.4
2024-03-05    21.7
              ... 
2024-05-25    20.1
2024-05-26    22.3
2024-05-27    18.4
2024-05-28    15.5
2024-05-29    20.4
Freq: D, Length: 90, dtype: float64

In [ ]:
print(s.describe())
print(clean.describe())

# 보정 전에 비해 보정 후 표준편차가 감소

count    90.000000
mean     19.797778
std       9.012687
min       7.500000
25%      15.800000
50%      18.100000
75%      21.300000
max      68.400000
dtype: float64
count    90.000000
mean     19.079321
std       6.368268
min       7.500000
25%      15.800000
50%      18.100000
75%      21.300000
max      37.823153
dtype: float64


④ 월별 집계와 이동평균

clean을 월 기준으로 그룹화해 관측일수·평균·최저·최고를 한 번에 구하세요. (groupby + agg)

7일 이동평균(rolling)을 구하고, 앞쪽 6개가 결측인 이유를 주석으로 적으세요.

기온 상위 5일을 정렬해 출력하세요.

전체 평균보다 높았던 날이 며칠인지 출력하세요.

In [47]:
monthly_summary = clean.groupby(clean.index.month).agg(
    관측일수='count',
    평균='mean',
    최저='min',
    최고='max'
)
monthly_summary

,관측일수,평균,최저,최고
3,31,18.895042,7.5,37.823153
4,30,19.077438,12.3,37.823153
5,29,19.278257,8.5,37.823153


In [48]:
rolling_mean_7d = clean.rolling(window=7).mean()
print(rolling_mean_7d.head(10))

# 이동평균 계산 시에는 최소 7일이 필요하기 때문에, 해당 기간보다 부족한 날의 값들은 모두 NaN값

2024-03-01          NaN
2024-03-02          NaN
2024-03-03          NaN
2024-03-04          NaN
2024-03-05          NaN
2024-03-06          NaN
2024-03-07    18.857143
2024-03-08    17.142857
2024-03-09    16.985714
2024-03-10    17.157143
Freq: D, dtype: float64


In [49]:
top5_hot_days = clean.sort_values(ascending=False).head(5)  # 내림차순으로 정렬 후 첫번째서부터 5개 도출
top5_hot_days

2024-03-26    37.823153
2024-03-15    37.823153
2024-05-12    37.823153
2024-05-14    37.823153
2024-05-15    37.823153
dtype: float64

In [50]:
count_above_mean = len(clean[clean > clean.mean()])
count_above_mean

37

## 2. 300건 규모의 온라인 쇼핑몰 주문 데이터를 생성해 할인/배송비 정책을 반영한 정산표 만들기

가상의 쇼핑몰 주문 데이터를 직접 만들고, 실제 정산 로직을 적용하는 과제입니다.

① 랜덤 주문 데이터 생성

np.random.seed(7), 주문 건수 300건으로 설정하세요.

아래 규칙으로 열을 만들어 DataFrame df를 생성하고, 주문번호를 인덱스로 지정하세요.

주문번호: ORD0001 ~ ORD0300 (4자리 0채움)

주문일: 2024-01-01 기준 0~179일 사이 랜덤 경과일

회원등급: 일반 50%, 실버 25%, 골드 15%, VIP 10% 확률로 추출

카테고리: 식품·의류·가전·도서·뷰티 중 랜덤

수량: 1~5 사이 랜덤 정수

단가: 5000·12000·25000·48000·99000 중 랜덤

쿠폰사용: True 30%, False 70%

head(), info(), describe()를 출력하고, 회원등급별 건수를 세어 의도한 비율과 비슷한지 확인하세요.

입력 누락 상황을 만듭니다. 랜덤한 20건의 수량을 결측값으로 바꾼 뒤, 결측 개수를 확인하고 중앙값으로 채우세요.

In [51]:
np.random.seed(7)
n_orders = 300

In [52]:
order_ids = [f'ORD{i:04d}' for i in range(1, n_orders + 1)]

start_date = pd.to_datetime('2024-01-01')
random_days = np.random.randint(0, 180, size=n_orders)
order_dates = start_date + pd.to_timedelta(random_days, unit='D')

grades = ['일반', '실버', '골드', 'VIP']
grade_probs = [0.50, 0.25, 0.15, 0.10]
member_grades = np.random.choice(grades, size=n_orders, p=grade_probs)

categories = ['식품', '의류', '가전', '도서', '뷰티']
order_categories = np.random.choice(categories, size=n_orders)

quantities = np.random.randint(1, 6, size=n_orders)

prices = [5000, 12000, 25000, 48000, 99000]
unit_prices = np.random.choice(prices, size=n_orders)

coupon_used = np.random.choice([True, False], size=n_orders, p=[0.30, 0.70])

In [54]:
df = pd.DataFrame({
    '주문일': order_dates,
    '회원등급': member_grades,
    '카테고리': order_categories,
    '수량': quantities,
    '단가': unit_prices,
    '쿠폰사용': coupon_used
}, index=order_ids)

df.index.name = '주문번호'

df

,주문일,회원등급,카테고리,수량,단가,쿠폰사용
주문번호,,,,,,
ORD0001,2024-06-24,일반,도서,4,12000,True
ORD0002,2024-01-26,일반,식품,3,5000,False
ORD0003,2024-03-08,일반,식품,1,12000,True
ORD0004,2024-05-31,실버,의류,1,12000,True
ORD0005,2024-04-13,일반,가전,4,12000,False
...,...,...,...,...,...,...
ORD0296,2024-05-10,VIP,식품,4,48000,False
ORD0297,2024-02-07,실버,의류,5,99000,True
ORD0298,2024-03-27,VIP,식품,3,48000,True


In [56]:
print(df.head())
print("\n")
df.info()
print("\n")
print(df.describe())

               주문일 회원등급 카테고리  수량     단가   쿠폰사용
주문번호                                          
ORD0001 2024-06-24   일반   도서   4  12000   True
ORD0002 2024-01-26   일반   식품   3   5000  False
ORD0003 2024-03-08   일반   식품   1  12000   True
ORD0004 2024-05-31   실버   의류   1  12000   True
ORD0005 2024-04-13   일반   가전   4  12000  False


<class 'pandas.DataFrame'>
Index: 300 entries, ORD0001 to ORD0300
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   주문일     300 non-null    datetime64[us]
 1   회원등급    300 non-null    str           
 2   카테고리    300 non-null    str           
 3   수량      300 non-null    int32         
 4   단가      300 non-null    int64         
 5   쿠폰사용    300 non-null    bool          
dtypes: bool(1), datetime64[us](1), int32(1), int64(1), str(2)
memory usage: 13.2+ KB


                       주문일          수량            단가
count                  300  300.000000    300.000000
mean   2024-03-31 09:45:3

In [58]:
grade_counts = df['회원등급'].value_counts()
grade_percentages = df['회원등급'].value_counts(normalize=True) * 100
print(grade_counts)
print("\n")
print(grade_percentages)

회원등급
일반     154
실버      63
골드      47
VIP     36
Name: count, dtype: int64


회원등급
일반     51.333333
실버     21.000000
골드     15.666667
VIP    12.000000
Name: proportion, dtype: float64


In [59]:
nan_indices = np.random.choice(len(df), size=20, replace=False)
df.iloc[nan_indices, df.columns.get_loc('수량')] = np.nan
df['수량'].isnull().sum()

np.int64(20)

In [60]:
quantity_median = df['수량'].median()
df['수량'] = df['수량'].fillna(quantity_median)
df['수량'].isnull().sum()

np.int64(0)

② 정산 로직을 파생 열로 구현

주문금액 = 수량 × 단가

등급할인율 = 일반 0%, 실버 3%, 골드 5%, VIP 10% (딕셔너리 + map 사용)

쿠폰할인율 = 쿠폰 사용 시 5%, 아니면 0% (np.where 사용)

총할인율 = 등급할인율 + 쿠폰할인율

할인금액 = 주문금액 × 총할인율 ÷ 100 (반올림)

배송비 = (주문금액 − 할인금액)이 30,000원 이상이면 0원, 아니면 3,000원

최종결제금액 = 주문금액 − 할인금액 + 배송비

주문규모 = 최종결제금액이 20만 이상 '대형', 5만 이상 '중형', 그 외 '소형' (사용자 정의 함수 + apply 사용)

In [61]:
# 1. 주문금액 계산 (수량 × 단가)
df['주문금액'] = df['수량'] * df['단가']

# 2. 등급할인율 계산 (딕셔너리 + map 사용)
grade_discount_map = {'일반': 0.00, '실버': 0.03, '골드': 0.05, 'VIP': 0.10}
df['등급할인율'] = df['회원등급'].map(grade_discount_map)

# 3. 쿠폰할인율 계산 (np.where 사용)
# 쿠폰사용이 True이면 5%(0.05), False이면 0%(0.00)
df['쿠폰할인율'] = np.where(df['쿠폰사용'] == True, 0.05, 0.00)

# 4. 총할인율 계산 (등급할인율 + 쿠폰할인율)
df['총할인율'] = df['등급할인율'] + df['쿠폰할인율']

# 5. 할인금액 계산 (주문금액 × 총할인율), 정수형으로 깔끔하게 반올림(.round(0))
df['할인금액'] = (df['주문금액'] * df['총할인율']).round(0)

# 6. 배송비 계산 (주문금액 − 할인금액이 30,000원 이상이면 0원, 아니면 3,000원)
# np.where을 활용하여 조건 처리
df['배송비'] = np.where((df['주문금액'] - df['할인금액']) >= 30000, 0, 3000)

# 7. 최종결제금액 계산 (주문금액 − 할인금액 + 배송비)
df['최종결제금액'] = df['주문금액'] - df['할인금액'] + df['배송비']

# 8. 주문규모 구분 (사용자 정의 함수 + apply 사용)
def classify_order_size(amount):
    if amount >= 200000:
        return '대형'
    elif amount >= 50000:
        return '중형'
    else:
        return '소형'

df['주문규모'] = df['최종결제금액'].apply(classify_order_size)

print(df.head())

               주문일 회원등급 카테고리   수량     단가   쿠폰사용     주문금액  등급할인율  쿠폰할인율  총할인율  \
주문번호                                                                           
ORD0001 2024-06-24   일반   도서  4.0  12000   True  48000.0   0.00   0.05  0.05   
ORD0002 2024-01-26   일반   식품  3.0   5000  False  15000.0   0.00   0.00  0.00   
ORD0003 2024-03-08   일반   식품  1.0  12000   True  12000.0   0.00   0.05  0.05   
ORD0004 2024-05-31   실버   의류  1.0  12000   True  12000.0   0.03   0.05  0.08   
ORD0005 2024-04-13   일반   가전  4.0  12000  False  48000.0   0.00   0.00  0.00   

           할인금액   배송비   최종결제금액 주문규모  
주문번호                                 
ORD0001  2400.0     0  45600.0   소형  
ORD0002     0.0  3000  18000.0   소형  
ORD0003   600.0  3000  14400.0   소형  
ORD0004   960.0  3000  14040.0   소형  
ORD0005     0.0     0  48000.0   소형  


③ 집계와 조건 분석

주문일에서 월을 뽑아 월 열을 추가하세요.

월별 주문 건수·매출 합계·평균 주문금액을 한 번에 구하세요.

카테고리별 × 회원등급별 매출 합계를 구하세요.

VIP 회원이면서 최종결제금액 10만 원 이상인 주문 건수를 구하세요.

쿠폰을 사용한 가전 카테고리 주문만 조회하세요.

카테고리별 매출 상위 2건씩을 뽑아 출력하세요.

In [62]:
# 1. 주문일에서 월을 뽑아 '월' 열 추가
df['월'] = df['주문일'].dt.month

# 2. 월별 주문 건수, 월별 매출(주문금액) 합계·평균 주문금액 요약
monthly_summary = df.groupby('월').agg(
    주문건수=('주문금액', 'count'),
    매출합계=('주문금액', 'sum'),
    평균주문금액=('주문금액', 'mean')
)

# 3. 카테고리별 × 회원등급별 매출(주문금액) 합계
category_grade_sum = df.groupby(['카테고리', '회원등급'])['주문금액'].sum()

# 4. VIP 회원이면서 최종결제금액 10만 원 이상인 주문 건수
vip_high_value_condition = (df['회원등급'] == 'VIP') & (df['최종결제금액'] >= 100000)
vip_high_value_count = len(df[vip_high_value_condition])

# 5. 쿠폰을 사용한 '가전' 카테고리 주문만 조회
coupon_home_appliances = df[(df['쿠폰사용'] == True) & (df['카테고리'] == '가전')]

# 6. 카테고리별 매출(주문금액) 상위 2건씩 뽑아 출력
top2_per_category = df.sort_values(by=['카테고리', '주문금액'], ascending=[True, False])
top2_summary = top2_per_category.groupby('카테고리').head(2)
top2_summary


,주문일,회원등급,카테고리,수량,단가,쿠폰사용,주문금액,등급할인율,쿠폰할인율,총할인율,할인금액,배송비,최종결제금액,주문규모,월
주문번호,,,,,,,,,,,,,,,
ORD0057,2024-03-08,일반,가전,5.0,99000,False,495000.0,0.00,0.00,0.00,0.0,0,495000.0,대형,3
ORD0216,2024-03-10,일반,가전,5.0,99000,False,495000.0,0.00,0.00,0.00,0.0,0,495000.0,대형,3
ORD0164,2024-06-17,일반,도서,4.0,99000,False,396000.0,0.00,0.00,0.00,0.0,0,396000.0,대형,6
ORD0196,2024-02-12,일반,도서,4.0,99000,True,396000.0,0.00,0.05,0.05,19800.0,0,376200.0,대형,2
ORD0071,2024-06-27,일반,뷰티,5.0,99000,False,495000.0,0.00,0.00,0.00,0.0,0,495000.0,대형,6
ORD0268,2024-01-19,일반,뷰티,4.0,99000,True,396000.0,0.00,0.05,0.05,19800.0,0,376200.0,대형,1
ORD0008,2024-01-24,일반,식품,5.0,99000,False,495000.0,0.00,0.00,0.00,0.0,0,495000.0,대형,1
ORD0171,2024-02-04,실버,식품,5.0,99000,False,495000.0,0.03,0.00,0.03,14850.0,0,480150.0,대형,2
ORD0297,2024-02-07,실버,의류,5.0,99000,True,495000.0,0.03,0.05,0.08,39600.0,0,455400.0,대형,2


④ 정리와 저장

중간 계산용 열 등급할인율, 쿠폰할인율을 drop으로 한 번에 삭제하세요.

결과를 orders.csv로 저장하세요. (엑셀에서 한글이 깨지지 않도록)

저장한 파일을 다시 읽되 주문번호를 인덱스로, 주문일을 날짜형으로 읽어오세요.

loc['ORD0010']과 iloc[9] 결과를 비교하고, 두 값이 같은 이유를 주석으로 적으세요.

In [63]:
df = df.drop(['등급할인율', '쿠폰할인율'], axis=1)
df.to_csv('orders.csv', encoding='utf-8-sig')

In [64]:
df_new = pd.read_csv('orders.csv', index_col='주문번호', parse_dates=['주문일'])
df_new

,주문일,회원등급,카테고리,수량,단가,쿠폰사용,주문금액,총할인율,할인금액,배송비,최종결제금액,주문규모,월
주문번호,,,,,,,,,,,,,
ORD0001,2024-06-24,일반,도서,4.0,12000,True,48000.0,0.05,2400.0,0,45600.0,소형,6
ORD0002,2024-01-26,일반,식품,3.0,5000,False,15000.0,0.00,0.0,3000,18000.0,소형,1
ORD0003,2024-03-08,일반,식품,1.0,12000,True,12000.0,0.05,600.0,3000,14400.0,소형,3
ORD0004,2024-05-31,실버,의류,1.0,12000,True,12000.0,0.08,960.0,3000,14040.0,소형,5
ORD0005,2024-04-13,일반,가전,4.0,12000,False,48000.0,0.00,0.0,0,48000.0,소형,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ORD0296,2024-05-10,VIP,식품,4.0,48000,False,192000.0,0.10,19200.0,0,172800.0,중형,5
ORD0297,2024-02-07,실버,의류,5.0,99000,True,495000.0,0.08,39600.0,0,455400.0,대형,2
ORD0298,2024-03-27,VIP,식품,3.0,48000,True,144000.0,0.15,21600.0,0,122400.0,중형,3


In [ ]:
print(df_new.loc['ORD0010'])      # 명시적 인덱스(ORD0010) 내용 출력  # '10번째'
print("\n")
print(df_new.iloc[9])             # 묵시적 인덱스(9) 내용 출력  # 인덱스 0부터 시작하므로 9는 '10번째'

# 해당 자료의 행을 삭제한 적이 없으므로, 명시적인덱스와 묵시적 인덱스 내용이 같을 수 있음

주문일       2024-03-30 00:00:00
회원등급                       실버
카테고리                       도서
수량                        4.0
단가                      48000
쿠폰사용                    False
주문금액                 192000.0
총할인율                     0.03
할인금액                   5760.0
배송비                         0
최종결제금액               186240.0
주문규모                       중형
월                           3
Name: ORD0010, dtype: object


주문일       2024-03-30 00:00:00
회원등급                       실버
카테고리                       도서
수량                        4.0
단가                      48000
쿠폰사용                    False
주문금액                 192000.0
총할인율                     0.03
할인금액                   5760.0
배송비                         0
최종결제금액               186240.0
주문규모                       중형
월                           3
Name: ORD0010, dtype: object


## 3. 4개 분기 지점별 실적 데이터를 생성/연결해 Multiindex 집계와 성장률 리포트 만들기

① 분기 데이터 생성 함수 만들고 연결하기

np.random.seed(99)로 고정하세요.

지점 5곳(서울·부산·대구·광주·대전) × 상품 3종(A·B·C)의 조합을 pd.MultiIndex.from_product로 만들고, 인덱스명을 지점, 상품으로 지정하세요.

분기명을 인자로 받아 아래 열을 가진 DataFrame을 반환하는 함수 make_quarter(q) 를 작성하세요.

판매량: 50~299 랜덤 정수 / 단가: 10000·15000·20000 중 랜덤 / 반품: 0~19 랜덤 정수 / 분기: 인자로 받은 값

이 함수로 1Q~4Q 데이터를 만들어 pd.concat으로 세로로 연결한 sales를 만드세요. (총 60행)

shape, index.names, head()로 구조를 확인하세요.

집계 누락 상황을 만듭니다. 랜덤한 8개 행의 판매량을 결측값으로 바꾼 뒤, 지점별 중앙값으로 채우세요. (groupby + transform)

In [78]:
df = pd.DataFrame()

In [82]:
np.random.seed(99)
branches = ["서울", "부산", "대구", "광주", "대전"]
products = ["A", "B", "C"]
m_idx = pd.MultiIndex.from_product([branches, products], names=["지점", "상품"])
m_idx

MultiIndex([('서울', 'A'),
            ('서울', 'B'),
            ('서울', 'C'),
            ('부산', 'A'),
            ('부산', 'B'),
            ('부산', 'C'),
            ('대구', 'A'),
            ('대구', 'B'),
            ('대구', 'C'),
            ('광주', 'A'),
            ('광주', 'B'),
            ('광주', 'C'),
            ('대전', 'A'),
            ('대전', 'B'),
            ('대전', 'C')],
           names=['지점', '상품'])

In [83]:
def make_quarter(q):
    data = {
        '판매량': np.random.randint(50, 300, len(m_idx)),
        '단가':   np.random.choice([10000, 15000, 20000], len(m_idx)),
        '반품':   np.random.randint(0, 20, len(m_idx)),
    }
    df = pd.DataFrame(data, index=m_idx)
    df['분기'] = q
    return df

In [84]:
quarters = ["1Q", "2Q", "3Q", "4Q"]
sales = pd.concat([make_quarter(q) for q in quarters])

In [86]:
print(sales.shape)
print("\n")
print(sales.index.names)
print("\n")
print(sales.head())

(60, 4)


['지점', '상품']


       판매량     단가  반품  분기
지점 상품                    
서울 A   179  15000  14  1Q
   B    85  10000   4  1Q
   C   235  15000  12  1Q
부산 A   218  20000  17  1Q
   B   251  10000   9  1Q


In [88]:
nan_locs = np.random.choice(len(sales), size=8, replace=False)
sales.iloc[nan_locs, sales.columns.get_loc("판매량")] = np.nan

branch_median = sales.groupby("지점")["판매량"].transform("median")
sales["판매량"] = sales["판매량"].fillna(branch_median)

sales["판매량"].isnull().sum()

np.int64(0)

② 파생 열과 MultiIndex 집계

매출 = 판매량 × 단가, 반품율 = 반품 ÷ 판매량 × 100 (소수 둘째 자리)

등급 열: 매출 400만 이상 'A', 200만 이상 'B', 그 외 'C' (np.where 중첩)

지점 기준으로 매출의 건수·합계·평균·최댓값을 한 번에 구하세요.

지점 × 상품 기준, 분기 × 지점 기준 매출 합계를 각각 구하세요.

등급별 행 개수를 구하세요.

In [89]:
sales["매출"] = sales["판매량"] * sales["단가"]
sales["반품율"] = np.round(sales["반품"] / sales["판매량"] * 100, 2)

sales["등급"] = np.where(sales["매출"] >= 4000000, "A", np.where(sales["매출"] >= 2000000, "B", "C"))

In [90]:
agg_branch = sales.groupby("지점")["매출"].agg(["count", "sum", "mean", "max"])
agg_branch

,count,sum,mean,max
지점,,,,
광주,12,28220000.0,2.351667e+06,4220000.0
대구,12,32945000.0,2.745417e+06,4740000.0
대전,12,37295000.0,3.107917e+06,5540000.0
부산,12,28490000.0,2.374167e+06,5640000.0
서울,12,28310000.0,2.359167e+06,4300000.0


In [91]:
sum_branch_product = sales.groupby(["지점", "상품"])["매출"].sum()

sum_quarter_branch = sales.groupby(["분기", "지점"])["매출"].sum()

print(sum_branch_product)
print("\n")
print(sum_quarter_branch)

지점  상품
광주  A     11690000.0
    B      6750000.0
    C      9780000.0
대구  A     10110000.0
    B     12230000.0
    C     10605000.0
대전  A     11205000.0
    B     15245000.0
    C     10845000.0
부산  A      9785000.0
    B      8060000.0
    C     10645000.0
서울  A      8415000.0
    B      9095000.0
    C     10800000.0
Name: 매출, dtype: float64


분기  지점
1Q  광주     6880000.0
    대구     6960000.0
    대전     9605000.0
    부산    12510000.0
    서울     5695000.0
2Q  광주     6420000.0
    대구     5995000.0
    대전    13380000.0
    부산     6075000.0
    서울     6595000.0
3Q  광주     6470000.0
    대구    11215000.0
    대전     7830000.0
    부산     4440000.0
    서울     8590000.0
4Q  광주     8450000.0
    대구     8775000.0
    대전     6480000.0
    부산     5465000.0
    서울     7430000.0
Name: 매출, dtype: float64


In [92]:
grade_counts = sales["등급"].value_counts()
grade_counts

등급
B    29
C    22
A     9
Name: count, dtype: int64

③ 순위와 교차표(피벗)

지점별 연간 매출 합계를 내림차순 정렬하고, rank로 순위를 매기세요.

pivot_table로 행=지점, 열=분기, 값=매출 합계인 교차표 pv를 만드세요.

pv에 연간합계 열(행 방향 합계, axis=1)을 추가하고 내림차순 정렬하세요.

성장률 열 = (4Q − 1Q) ÷ 1Q × 100 (소수 첫째 자리)을 추가하고, 성장률 순으로 정렬해 출력하세요.

In [93]:
branch_total = sales.groupby("지점")["매출"].sum().reset_index()
branch_total = branch_total.sort_values(by="매출", ascending=False)
branch_total["순위"] = (
    branch_total["매출"].rank(ascending=False, method="min")
)


In [94]:
pv = sales.pivot_table(index="지점", columns="분기", values="매출", aggfunc="sum")


In [95]:
pv["연간합계"] = pv.sum(axis=1)
pv = pv.sort_values(by="연간합계", ascending=False)
pv

분기,1Q,2Q,3Q,4Q,연간합계
지점,,,,,
대전,9605000.0,13380000.0,7830000.0,6480000.0,37295000.0
대구,6960000.0,5995000.0,11215000.0,8775000.0,32945000.0
부산,12510000.0,6075000.0,4440000.0,5465000.0,28490000.0
서울,5695000.0,6595000.0,8590000.0,7430000.0,28310000.0
광주,6880000.0,6420000.0,6470000.0,8450000.0,28220000.0


In [99]:
pv["성장률"] = np.round((pv["4Q"] - pv["1Q"]) / pv["1Q"] * 100, 1)
pv_sorted_by_growth = pv.sort_values(by="성장률", ascending=False)
pv_sorted_by_growth

분기,1Q,2Q,3Q,4Q,연간합계,성장률
지점,,,,,,
서울,5695000.0,6595000.0,8590000.0,7430000.0,28310000.0,30.5
대구,6960000.0,5995000.0,11215000.0,8775000.0,32945000.0,26.1
광주,6880000.0,6420000.0,6470000.0,8450000.0,28220000.0,22.8
대전,9605000.0,13380000.0,7830000.0,6480000.0,37295000.0,-32.5
부산,12510000.0,6075000.0,4440000.0,5465000.0,28490000.0,-56.3


④ 가로 방향 연결과 저장

1Q 지점별 매출 합계와 4Q 지점별 매출 합계를 각각 Series로 만들고, pd.concat의 axis=1로 나란히 붙여 비교표를 만드세요. (keys로 열 이름 지정)

지점별 매출 1위 주문 행을 각각 1건씩 뽑아 출력하세요.

결과를 sales_report.csv로 저장한 뒤, 지점과 상품을 함께 인덱스로 지정해 다시 읽어오세요.

읽어온 데이터에서 loc['서울']로 서울 지점 데이터만 조회하고, 원본 sales와 인덱스 구조가 어떻게 달라졌는지 주석으로 적으세요.

In [100]:
s_1q = sales[sales["분기"] == "1Q"].groupby("지점")["매출"].sum()
s_4q = sales[sales["분기"] == "4Q"].groupby("지점")["매출"].sum()

comparison_df = pd.concat([s_1q, s_4q], axis=1, keys=["1Q_매출", "4Q_매출"])
comparison_df

,1Q_매출,4Q_매출
지점,,
광주,6880000.0,8450000.0
대구,6960000.0,8775000.0
대전,9605000.0,6480000.0
부산,12510000.0,5465000.0
서울,5695000.0,7430000.0


In [103]:
top = sales.sort_values(by='매출', ascending=False).groupby('지점').head(1)
top

,,판매량,단가,반품,분기,매출,반품율,등급
지점,상품,,,,,,,
부산,C,282.0,20000,0,1Q,5640000.0,0.00,A
대전,B,277.0,20000,5,2Q,5540000.0,1.81,A
대구,B,237.0,20000,6,3Q,4740000.0,2.53,A
서울,B,215.0,20000,18,3Q,4300000.0,8.37,A
광주,A,211.0,20000,11,2Q,4220000.0,5.21,A


In [104]:
top.to_csv("sales_report.csv", encoding="utf-8-sig")

In [105]:
df_report = pd.read_csv("sales_report.csv", index_col=["지점", "상품"])

In [ ]:
seoul_data = df_report.loc["서울"]
seoul_data

# 인덱스 구성의 차이:
# 원본 sales: 1Q~4Q 전체 분기의 모든 지점·상품 데이터가 포함된 총 60개 행입니다.  (Multiindex)
# sales_report.csv (지점별 1위만 저장한 경우): 지점당 1건씩 총 5개 행만 존재하므로,
# seoul_data를 조회하면 서울 지점의 1위 상품에 해당하는 딱 '1개 행'만 출력됩니다.   (단일 인덱스)

,판매량,단가,반품,분기,매출,반품율,등급
상품,,,,,,,
B,215.0,20000,18,3Q,4300000.0,8.37,A
